In [ ]:
%pip install openpyxl xlrd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [ ]:
iwpc = pd.read_excel("../data/IWPC_Data.xls", sheet_name="Subject Data")
meta = pd.read_excel("../data/IWPC_Data.xls", sheet_name="Metadata")
iwpc.shape

## The data
5,700 patients from the International Warfarin Pharmacogenetics Consortium.
One row per patient; 68 columns. Three things to notice: age is stored as a
text range, the genotypes are categorical, and there are missing values
throughout.

In [ ]:
iwpc.head()
list(iwpc.columns)
meta.head(20)

In [ ]:
dose = iwpc["Therapeutic Dose of Warfarin"].dropna()
dose.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(dose, bins=50);          axes[0].set_title("dose (mg/week)")
axes[1].hist(np.sqrt(dose), bins=50); axes[1].set_title("√dose")
axes[2].hist(np.log(dose), bins=50);  axes[2].set_title("log(dose)")
plt.show()

print("skew, dose: ", dose.skew())
print("skew, √dose:", np.sqrt(dose).skew())
print("skew, log:  ", np.log(dose).skew())

### Filtering patients

Two columns decide who is usable. Neither can be a *feature* — both are known
only after treatment — but they can select rows:

- **Reached a stable dose.** If a patient never settled, their recorded dose
  is not a trustworthy target.
- **Dose actually recorded.** No $y$, nothing to learn from.

$$5700 \;\xrightarrow{\text{both filters}}\; 5410
      \;\xrightarrow{\text{drop 39 missing ages (next cell)}}\; 5371$$

In [ ]:
stable   = iwpc["Subject Reached Stable Dose of Warfarin"] == 1
has_dose = iwpc["Therapeutic Dose of Warfarin"].notna()
keep     = stable & has_dose

iwpc_clean = iwpc[keep]
iwpc_clean.shape

In [ ]:
### What needs cleaning
Before encoding anything, check two things for every candidate feature:
how much is missing, and what values the categorical columns actually take.

In [ ]:
cols = ["Age", "Height (cm)", "Weight (kg)",
        "VKORC1 -1639 consensus", "CYP2C9 consensus", "Race (OMB)",
        "Carbamazepine (Tegretol)", "Phenytoin (Dilantin)",
        "Rifampin or Rifampicin", "Amiodarone (Cordarone)"]

iwpc_clean[cols].isna().sum()

for c in ["VKORC1 -1639 consensus", "CYP2C9 consensus", "Race (OMB)"]:
    print(iwpc_clean[c].value_counts(dropna=False), "\n")

for c in ["Carbamazepine (Tegretol)", "Phenytoin (Dilantin)",
          "Rifampin or Rifampicin", "Amiodarone (Cordarone)"]:
    print(iwpc_clean[c].value_counts(dropna=False), "\n")    

In [33]:
# Step 3: drop patients with no age (only 39)
data = iwpc_clean.dropna(subset=["Age"]).copy()

# Age: "60 - 69" -> 6   (first character = the decade)
data["age_decades"] = pd.to_numeric(data["Age"].str[0])

# VKORC1: reference = G/G; missing -> "Unknown"
vk = data["VKORC1 -1639 consensus"].fillna("Unknown")
data["vkorc1_AG"]      = (vk == "A/G").astype(int)
data["vkorc1_AA"]      = (vk == "A/A").astype(int)
data["vkorc1_unknown"] = (vk == "Unknown").astype(int)

# CYP2C9: reference = *1/*1; missing or rare alleles -> unknown
cyp = data["CYP2C9 consensus"]
cyp_map = {"*1/*2": "cyp_12", "*1/*3": "cyp_13", "*2/*2": "cyp_22",
           "*2/*3": "cyp_23", "*3/*3": "cyp_33"}
for genotype, name in cyp_map.items():
    data[name] = (cyp == genotype).astype(int)
data["cyp_unknown"] = (~cyp.isin(list(cyp_map) + ["*1/*1"])).astype(int)

# Race: reference = White
race = data["Race (OMB)"]
data["race_asian"]   = (race == "Asian").astype(int)
data["race_black"]   = (race == "Black or African American").astype(int)
data["race_unknown"] = (race == "Unknown").astype(int)

In [68]:
inducers = ["Carbamazepine (Tegretol)", "Phenytoin (Dilantin)", "Rifampin or Rifampicin"]
data["enzyme_inducer"] = (data[inducers].fillna(0) == 1).any(axis=1).astype(int)
data["amiodarone"]     = (data["Amiodarone (Cordarone)"].fillna(0) == 1).astype(int)
print(data["enzyme_inducer"].sum(), data["amiodarone"].sum())

54 246


In [ ]:
features = ["age_decades", "Height (cm)", "Weight (kg)",
            "vkorc1_AG", "vkorc1_AA", "vkorc1_unknown",
            "cyp_12", "cyp_13", "cyp_22", "cyp_23", "cyp_33", "cyp_unknown",
            "race_asian", "race_black", "race_unknown",
            "enzyme_inducer", "amiodarone"]

X_df = data[features]
y = np.sqrt(data["Therapeutic Dose of Warfarin"]).to_numpy()   # y ∈ ℝⁿ

print(X_df.shape, y.shape)
print(X_df.isna().sum())

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=0)

print(X_train_df.shape, X_test_df.shape)

### Imputing height and weight

Height is missing for 19% of patients and weight for 5% — too many to drop.
Both are filled with the **training** median.

The median is a *learned parameter*, not a fixed constant, so it belongs to the
training set. Computing it over the test set would let test patients inform
each other's predictions, and would not match deployment: a clinician with one
new patient has no test set to average over. The rule for the rest of this
notebook is the same every time — **learn on train, apply to both**.

In [ ]:
medians = X_train_df.median()          # computed from TRAINING patients only

X_train_df = X_train_df.fillna(medians)
X_test_df  = X_test_df.fillna(medians)  # test uses the TRAINING medians

print(X_train_df.isna().sum().sum(), X_test_df.isna().sum().sum())   # both should be 0

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)   # learn x̄ and s, then apply
X_test  = scaler.transform(X_test_df)        # apply the SAME x̄ and s

print(X_train.shape, X_test.shape)
print(X_train.mean(axis=0).round(3)[:3], X_train.std(axis=0).round(3)[:3])

In [69]:
n, p = X_train.shape

w      = np.zeros(p)     # w ∈ ℝ¹⁷
b      = 0.0             # scalar
eta    = 0.1
steps  = 2000
losses = []

for t in range(steps):
    y_hat  = X_train @ w + b              # ŷ = Xw + b1    → vector ∈ ℝⁿ
    r      = y_train - y_hat              # r = y − ŷ      → vector ∈ ℝⁿ
    losses.append(np.mean(r**2))          # L = (1/n) rᵀr  → scalar
    grad_w = -(2/n) * (X_train.T @ r)     # −(2/n) Xᵀr     → vector ∈ ℝ¹⁷
    grad_b = -(2/n) * np.sum(r)           # −(2/n) 1ᵀr     → scalar
    w      = w - eta * grad_w             # w ← w − η∇_w L
    b      = b - eta * grad_b             # b ← b − η ∂L/∂b

assert w.shape == (p,), w.shape

In [ ]:
# Standardised weights: units are "per standard deviation of that feature".
# Signs are directly interpretable; magnitudes are NOT comparable across
# features with different spreads (see Cell 21). Sorted by influence.
coef = pd.DataFrame({"feature": features, "w_std": w})
coef.sort_values("w_std", key=abs, ascending=False)

In [ ]:
plt.plot(losses)
plt.axhline(y_train.var(), color="red", ls="--", label="predicting the mean")
plt.yscale("log")
plt.xlabel("iteration"); plt.ylabel("MSE (√dose scale)"); plt.legend()
plt.show()

In [ ]:
lr = LinearRegression().fit(X_train, y_train)

print("sklearn w:", lr.coef_.round(3))
print("your w:   ", w.round(3))
print("max difference:", np.abs(lr.coef_ - w).max())
print("sklearn b:", round(lr.intercept_, 3))

In [ ]:
# predictions on the √ scale
y_pred_train = X_train @ w + b
y_pred_test  = X_test  @ w + b

mse_train = np.mean((y_train - y_pred_train)**2)
mse_test  = np.mean((y_test  - y_pred_test )**2)
r2_test   = 1 - mse_test / y_test.var()

print(f"train MSE {mse_train:.3f}   test MSE {mse_test:.3f}   test R² {r2_test:.3f}")

In [ ]:
dose_true = y_test**2              # mg/week
dose_pred = y_pred_test**2         # mg/week  ← the back-transform

mae  = np.mean(np.abs(dose_true - dose_pred))
pw20 = np.mean(np.abs(dose_true - dose_pred) <= 0.20 * dose_true)

print(f"MAE: {mae:.2f} mg/week")
print(f"within 20% of the true dose: {100*pw20:.1f}% of patients")

In [ ]:
import os
os.makedirs("../images", exist_ok=True)

res_sqrt = y_test - y_pred_test            # residuals on the √ scale
res_raw  = dose_true - dose_pred           # residuals in mg/week

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].hist(res_sqrt, bins=40)
ax[0].set_title("A: residuals, √ scale\n(bell-shaped?)")

ax[1].scatter(y_pred_test, res_sqrt, s=5, alpha=0.3)
ax[1].axhline(0, color="red")
ax[1].set_xlabel("predicted √dose"); ax[1].set_ylabel("residual")
ax[1].set_title("B: √ scale\n(even spread?)")

ax[2].scatter(dose_pred, res_raw, s=5, alpha=0.3)
ax[2].axhline(0, color="red")
ax[2].set_xlabel("predicted dose (mg/week)"); ax[2].set_ylabel("residual (mg/week)")
ax[2].set_title("C: raw mg/week scale\n(even spread?)")

plt.tight_layout()
plt.savefig("../images/residuals.png", dpi=150, bbox_inches="tight")   # BEFORE show
plt.show()

print("residual skew, √ scale:", pd.Series(res_sqrt).skew().round(3))
print("residual skew, raw    :", pd.Series(res_raw).skew().round(3))

bins = pd.qcut(y_pred_test, 4)
print(pd.Series(res_sqrt).groupby(bins).std().round(3))
print(pd.Series(res_raw ).groupby(bins).std().round(2))

In [ ]:
w_raw = w / scaler.scale_
b_raw = b - np.sum(w_raw * scaler.mean_)

for name, wr in zip(features, w_raw):
    print(f"{name:20s} {wr:+.4f}")
print(f"{'intercept':20s} {b_raw:+.4f}")

In [ ]:
X_test_raw = X_test_df.to_numpy()
print("max difference:", np.abs((X_test_raw @ w_raw + b_raw) - y_pred_test).max())

In [ ]:
# ---- Export the trained model -------------------------------------------
# A trained model is just these numbers: 17 weights, one bias, and the feature
# order they belong to. Saved to disk so the website can use them without
# re-running this notebook, and so they never get hand-copied out of date.

model = {
    "features": features,
    "w_raw":    [round(float(v), 6) for v in w_raw],   # float() — np.float64 is not JSON-serialisable
    "b_raw":    round(float(b_raw), 6),

    # sensible starting values for the prediction form
    "defaults": {
        "age_decades": float(X_train_df["age_decades"].median()),
        "height_cm":   float(X_train_df["Height (cm)"].median()),
        "weight_kg":   float(X_train_df["Weight (kg)"].median()),
    },

    # 1st–99th percentile of the training data; outside this the page warns
    # the visitor that the model is extrapolating
    "ranges": {
        "age_decades": [float(X_train_df["age_decades"].min()),
                        float(X_train_df["age_decades"].max())],
        "height_cm":   [float(X_train_df["Height (cm)"].quantile(0.01)),
                        float(X_train_df["Height (cm)"].quantile(0.99))],
        "weight_kg":   [float(X_train_df["Weight (kg)"].quantile(0.01)),
                        float(X_train_df["Weight (kg)"].quantile(0.99))],
    },

    # quoted on the page next to the prediction, so they cannot drift
    "metrics": {
        "n_train":     int(len(y_train)),
        "n_test":      int(len(y_test)),
        "r2_test":     round(float(r2_test), 3),
        "mae_mg_week": round(float(mae), 2),
        "pw20":        round(float(100 * pw20), 1),
        "train_mse":   round(float(mse_train), 3),
        "test_mse":    round(float(mse_test), 3),
    },

    # model card: what this was trained on and how
    "provenance": {
        "data":          "IWPC_Data.xls, 'Subject Data' sheet (ClinPGx downloads)",
        "n_total":       int(len(y)),
        "filters":       "reached stable dose; dose recorded; age recorded",
        "target":        "sqrt(dose in mg/week)",
        "test_size":     0.2,
        "random_state":  0,
        "imputation":    "training median for height and weight",
        "optimizer":     "full-batch gradient descent, eta=0.1, 2000 steps",
        "trained_on":    pd.Timestamp.today().strftime("%Y-%m-%d"),
        "not_for_clinical_use": True,
    },
}

os.makedirs("../model", exist_ok=True)
with open("../model/warfarin_model.json", "w") as f:
    json.dump(model, f, indent=2)          # dump → file;  dumps → string

print(f"saved {len(model['w_raw'])} weights to ../model/warfarin_model.json")
print(json.dumps(model["metrics"], indent=2))